In [1]:
import os, requests, time, re, shutil, logging, tempfile
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
from PyPDF2 import PdfReader
from datetime import datetime
import pdfplumber
from collections import defaultdict

# Configure logging and start timer
start = time.time()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
pd.set_option('display.max_rows', 500, 'display.max_columns', 500, 'display.max_colwidth', 1000)

# Setup directories
BASE_DIR = "teb_financial_data"
dirs = {k: os.path.join(BASE_DIR, k) for k in ['original', 'current', 'output']}
for d in ['current/balance-sheet', 'current/income-statement'] + list(dirs.values()):
    os.makedirs(d, exist_ok=True)
    print(f"📁 Created: {d}")

PROCESSED_FILES_LOG = os.path.join(BASE_DIR, "processed_files.csv")
bank_url = 'https://teb-kos.com/raportet-financiare/'

def load_processed_files():
    return set(pd.read_csv(PROCESSED_FILES_LOG)['file_name'].tolist()) if os.path.exists(PROCESSED_FILES_LOG) else set()

def save_processed_file(file_name, status=1, reporting_date=None, current_name=None, is_corrupt=0):
    processed = load_processed_files()
    if file_name not in processed:
        df = pd.DataFrame({'file_name': [file_name], 'processed_date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
                          'status': [status], 'reporting_date': [reporting_date], 'current_name': [current_name], 'is_corrupt': [is_corrupt]})
        (pd.concat([pd.read_csv(PROCESSED_FILES_LOG), df], ignore_index=True) if os.path.exists(PROCESSED_FILES_LOG) else df).to_csv(PROCESSED_FILES_LOG, index=False)

def scrape_pdf_links(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        pdf_links = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.lower().endswith('.pdf') and not href.lower().startswith('https://91.187.98.58'):
                pdf_links.append(urljoin('https://www.teb-kos.com' if not href.lower().startswith('https://') else '', href))
                print(f"   📄 Found: {os.path.basename(href)}")
        print(f"\n✅ Total PDFs: {len(pdf_links)}")
        return pdf_links
    except Exception as e:
        print(f"❌ Error: {e}")
        return []

def download_pdf(url, local_dir):
    file_name = os.path.basename(url)
    local_path = os.path.join(local_dir, file_name)
    try:
        response = requests.get(url)
        response.raise_for_status()
        with open(local_path, 'wb') as f: f.write(response.content)
        print(f"   ✅ Downloaded: {file_name} ({os.path.getsize(local_path)/1024:.1f} KB)")
        return local_path
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        return None

def extract_date(text, filename):
    match = re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', text) or re.search(r'\d{2}[./-]\d{2}[./-]\d{4}', filename)
    return match.group().replace('-', '.').replace('/', '.') if match else None

def download_all(force=False):
    processed = set() if force else load_processed_files()
    print(f"🔍 Scraping from: {bank_url}")
    pdf_links = scrape_pdf_links(bank_url)
    if not pdf_links: return []
    
    new_files, file_metadata = [], []
    Balance_sheet, Income_Statement = ['bilanc', 'bilancit', 'bsh', 'bilanci', 'bilance', 'balance'], ['is', 'ardhurave', 'income']
    categories = {'balance-sheet': Balance_sheet, 'income-statement': Income_Statement}
    
    for i, pdf_url in enumerate(pdf_links, 1):
        file_name = os.path.basename(pdf_url)
        print(f"\n[{i}/{len(pdf_links)}] Processing: {file_name}")
        if file_name in processed and not force:
            print("   ⏭️ Skipping (already processed)")
            continue
        new_files.append(file_name)
        
        file_path = download_pdf(pdf_url, dirs['original'])
        if not file_path: continue
        
        # Get metadata date
        dt = None
        try:
            with open(file_path, 'rb') as f:
                metadata = PdfReader(f).metadata
                for k, v in (metadata or {}).items():
                    if "/CreationDate" in k:
                        date_str = re.search(r'D:(\d+)', str(v))
                        if date_str: dt = datetime.strptime(date_str.group(1), '%Y%m%d%H%M%S')
        except: pass
        
        # Categorize
        file_lower = file_name.lower()
        found_category = next((cat for cat, keywords in categories.items() if any(k in file_lower for k in keywords)), None)
        
        if not found_category:
            try:
                with pdfplumber.open(file_path) as pdf:
                    text = pdf.pages[0].extract_text() or ""
                    if any(k in text.lower() for k in Balance_sheet): found_category = 'balance-sheet'
                    elif any(k in text.lower() for k in Income_Statement): found_category = 'income-statement'
            except: pass
        
        # Extract date from content
        date = None
        try:
            with pdfplumber.open(file_path) as pdf:
                full_text = "".join(page.extract_text() or "" for page in pdf.pages[:3])
                date = extract_date(full_text, file_name) or datetime.now().strftime('%d.%m.%Y')
        except: date = datetime.now().strftime('%d.%m.%Y')
        
        if found_category:
            new_name = f"teb_{'bs' if found_category=='balance-sheet' else 'is'}_{date}.pdf"
            shutil.copy2(file_path, os.path.join(dirs['current'], found_category, new_name))
            file_metadata.append({'file_name': file_name, 'current_name': new_name, 'category': found_category, 'reporting_date': date, 'creation_date': dt, 'is_corrupt': 0})
            print(f"   ✅ Categorized as {found_category} -> {new_name}")
        else:
            file_metadata.append({'file_name': file_name, 'current_name': None, 'category': 'uncategorized', 'reporting_date': date, 'creation_date': dt, 'is_corrupt': 0})
        
        save_processed_file(file_name, reporting_date=date, current_name=new_name if found_category else None)
    
    if file_metadata:
        pd.DataFrame(file_metadata).to_csv(os.path.join(dirs['output'], f"file_metadata_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"), index=False)
    
    return [m['current_name'] for m in file_metadata if m.get('current_name')]

def replace_negatives(val):
    """Convert parenthesized numbers to negative - FIXED"""
    if pd.isna(val) or str(val).strip() in ['', '-', '0']: return '0'
    val = str(val).strip()
    # Handle parentheses: (710) -> -710
    if '(' in val and ')' in val:
        # Remove parentheses and any commas, add minus sign
        num = val.replace('(', '').replace(')', '').replace(',', '').strip()
        return f"-{num}"
    return val

def extract_values(line):
    return re.findall(r'\(?[-\d,]+(?:\.\d+)?\)?', line)

def has_values(line):
    return bool(re.search(r'\d', line))

# Data storage
income_data, balance_data = defaultdict(list), defaultdict(list)

def process_pdf(pdf_path, target_categories, filename, is_income=True):
    file_current_name = os.path.basename(filename)
    data_dict = income_data if is_income else balance_data
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = "".join(page.extract_text() or "" for page in pdf.pages)
            dt_report = extract_date(full_text, pdf_path) or datetime.now().strftime('%d.%m.%Y')
            
            lines = full_text.split('\n')
            for i, line in enumerate(lines):
                line = line.strip()
                if not line: continue
                
                next_line = lines[i + 1].strip() if i + 1 < len(lines) else ""
                
                for category in target_categories:
                    if category.lower() in line.lower():
                        # Check for duplicates
                        if any(entry.get('Category') == category for entry in data_dict[dt_report]):
                            continue
                        
                        values = []
                        if has_values(line):
                            values = extract_values(line)
                        elif has_values(next_line) and not any(cat.lower() in next_line.lower() for cat in target_categories):
                            values = extract_values(next_line)
                        
                        if values:
                            curr_quarter = values[-1] if values else "0"
                            prev_quarter = values[-2] if len(values) > 1 else "0"
                            
                            data_dict[dt_report].append({
                                'Category': category,
                                'PREVIOUS_QUARTER': prev_quarter,
                                'CURRENT_QUARTER': curr_quarter,
                                'DT_REPORT': dt_report,
                                'FILE_NAME': file_current_name
                            })
        print(f"   ✅ Processed: {file_current_name}")
    except Exception as e:
        print(f"   ❌ Error: {e}")

# Define categories (same as before, but cleaned up)
income_statement_categories = [
    'Të hyrat nga interesi', 'Shpenzimet e interesit', 'Neto të hyrat nga interesi',
    'Të hyrat nga tarifat dhe komisionet', 'Shpenzimet e tarifave dhe komisioneve',
    'Neto të hyrat nga tarifat dhe komisionet', 'Neto të hyrat nga tregtimi me valuta të huaja',
    'Neto të hyrat nga tregtimi', 'Neto të hyrat nga instrumentet tjera financiare',
    'Neto të hyrat (shpenzimet) tjera operative', 'Gjithsej të hyrat',
    'Provizionet për humbjet nga kreditë', 'Provizione të tjera', 'Fitimi (humbja) para tatimit',
    'Fitimi/(humbja) para tatimit', 'Fitimi para tatimit', 'Shpenzimet e tatimit në fitim',
    'Fitimi (humbja) neto', 'Fitimi/(humbja) neto', 'Fitimi neto',
    'Të ardhurat tjera gjithëpërfshirëse', 'Gjithsej të ardhurat gjithëpërfshirëse',
    'Gjithsej te ardhurat gjithëpërfshirëse', 'Gjithsejt te ardhurat gjitheperfshirese per vitin',
    'Gjithsejt të ardhurat gjithëpërfshirëse për vitin', 'Fitimi neto pas tatimit',
    'Fitimi/(humbja) neto pas tatimit'
]

balance_sheet_categories = [
    "Paraja e gatshme dhe gjendja me BQK-në", "Kërkesat ndaj bankave", "Bonot e thesarit",
    "Mjetet jo qarkulluese të mbajtura për shitje", "Investimet në letra me vlerë",
    "Kreditë dhe paradhëniet ndaj klientëve", "Patundshmëritë dhe pajisjet",
    "Pasuritë e paprekshme", "Pasuritë tatimore të shtyra", "Pasuritë tjera",
    "Gjithsej pasuritë", "Depozitat e klientëve", "Detyrimet ndaj bankave",
    "Borgjet afatgjata", "Borxhet afataxhata", "Borxhet afatshkurtera",
    "Borxhet afatshkurtëra", "Borgjet afatshkurtera", "Borgjet afat shkurta",
    "Borgjet afatshkurta", "Borxhi i ndërvarur", "Fondet tjera të huazuara",
    "Detyrimet tatimore të shtyra", "Detyrimet tjera", "Gjithsej detyrimet",
    'Ekuiteti i aksionarëve', "Kapitali aksionar", "Fitimi i mbajtur nga vitet paraprake",
    "Humbja nga vitet paraprake", "Rezervat e kapitalit", "Fitimi i mbajtur/(humbja) nga vitet paraprake",
    "Rezerva per vleren e tregut", "Fitimi/(humbja) e vitit aktual", "Fitimi i vitit aktual",
    "Përbërësit tjerë të ekuititetit", 'Përbërësit tjerë të ekuitetit', 
    'Gjithsej ekuiteti i aksionarëve', "Gjithsej ekuititeti I aksionarëve",
    "Gjithsej detyrimet dhe ekuiteti i aksionarëve", "Gjithsej detyrimet dhe ekuititeti i aksionarëve"
]

def clean_numeric_values(df, prev_col, curr_col):
    """Clean and convert numeric columns - FIXED for parentheses"""
    for col in [prev_col, curr_col]:
        df[col] = df[col].astype(str).str.replace('-', '0').apply(replace_negatives)
        # Remove commas and convert to numeric
        df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='coerce').fillna(0)
    return df

def main(force=False, extract_only=False):
    print("\n" + "="*60)
    print("🏦 TEB BANK FINANCIAL DATA EXTRACTION")
    print("="*60)
    
    global income_data, balance_data
    income_data.clear(); balance_data.clear()
    
    # Get files to process
    if extract_only:
        new_files = []
        for category in ['balance-sheet', 'income-statement']:
            cat_dir = os.path.join(dirs['current'], category)
            if os.path.exists(cat_dir):
                new_files.extend([f for f in os.listdir(cat_dir) if f.endswith('.pdf')])
        print(f"\n📂 EXTRACT-ONLY: Found {len(new_files)} existing files")
    else:
        print("\n🌐 DOWNLOAD MODE")
        new_files = download_all(force=force)
    
    # Process files
    for category in ['balance-sheet', 'income-statement']:
        cat_dir = os.path.join(dirs['current'], category)
        if not os.path.exists(cat_dir): continue
        
        print(f"\n📂 Processing {category} files...")
        for filename in os.listdir(cat_dir):
            if filename in new_files or extract_only:
                file_path = os.path.join(cat_dir, filename)
                print(f"\n   🔍 Processing: {filename}")
                if category == 'income-statement':
                    process_pdf(file_path, income_statement_categories, filename, is_income=True)
                else:
                    process_pdf(file_path, balance_sheet_categories, filename, is_income=False)
    
    # Create DataFrames
    full_bs = pd.concat([pd.DataFrame(data) for data in balance_data.values()], ignore_index=True) if balance_data else pd.DataFrame()
    full_is = pd.concat([pd.DataFrame(data) for data in income_data.values()], ignore_index=True) if income_data else pd.DataFrame()
    
    # Process Balance Sheet
    if not full_bs.empty:
        full_bs.columns = ['BALANCE_CATEGORY', 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE', 'DT_REPORT', 'FILE_NAME']
        full_bs = clean_numeric_values(full_bs, 'PREVIOUS_BALANCE_VALUE', 'CURRENT_BALANCE_VALUE')
        
        # Category mapping for balance sheet
        bs_map = {
            "Paraja e gatshme dhe gjendja me BQK-në": "20", "Kërkesat ndaj bankave": "21", "Bonot e thesarit": "22",
            "Investimet në letra me vlerë": "23", "Mjetet jo qarkulluese të mbajtura për shitje": "24",
            "Kreditë dhe paradhëniet ndaj klientëve": "26", "Patundshmëritë dhe pajisjet": "27",
            "Pasuritë e paprekshme": "28", "Pasuritë tatimore të shtyra": "29", "Pasuritë tjera": "30",
            "Gjithsej pasuritë": "31", "Depozitat e klientëve": "32", "Detyrimet ndaj bankave": "33",
            "Borgjet afatgjata": "33", "Borxhet afataxhata": "33", "Borxhet afatshkurtera": "33",
            "Borxhet afatshkurtëra": "33", "Borgjet afatshkurtera": "33", "Borgjet afat shkurta": "33",
            "Borgjet afatshkurta": "33", "Borxhi i ndërvarur": "33", "Fondet tjera të huazuara": "34",
            "Detyrimet tatimore të shtyra": "35", "Detyrimet tjera": "36", "Gjithsej detyrimet": "37",
            "Kapitali aksionar": "38", "Rezervat e kapitalit": "40", "Fitimi i mbajtur/(humbja) nga vitet paraprake": "41",
            "Fitimi i mbajtur nga vitet paraprake": "41", "Humbja nga vitet paraprake": "41",
            "Fitimi/(humbja) e vitit aktual": "42", "Fitimi i vitit aktual": "42",
            "Përbërësit tjerë të ekuititetit": "43", "Përbërësit tjerë të ekuitetit": "43",
            "Gjithsej ekuiteti i aksionarëve": "44", "Ekuiteti i aksionarëve": "44",
            "Gjithsej detyrimet dhe ekuiteti i aksionarëve": "45"
        }
        full_bs['CATEGORY_CODE'] = full_bs['BALANCE_CATEGORY'].map(bs_map)
        full_bs['STATEMENT_TYPE'] = 'BALANCE_SHEET'
        full_bs['DT_REPORT'] = pd.to_datetime(full_bs['DT_REPORT'], format='%d.%m.%Y', errors='coerce')
    
    # Process Income Statement
    if not full_is.empty:
        full_is.columns = ['INCOME_CATEGORY', 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE', 'DT_REPORT', 'FILE_NAME']
        full_is = clean_numeric_values(full_is, 'PREVIOUS_INCOME_VALUE', 'CURRENT_INCOME_VALUE')
        
        # Category mapping for income statement
        is_map = {
            'Të hyrat nga interesi': '1', 'Shpenzimet e interesit': '2', 'Neto të hyrat nga interesi': '3',
            'Të hyrat nga tarifat dhe komisionet': '4', 'Shpenzimet e tarifave dhe komisioneve': '5',
            'Neto të hyrat nga tarifat dhe komisionet': '6', 'Neto të hyrat nga tregtimi me valuta të huaja': '7',
            'Neto të hyrat nga tregtimi': '7', 'Neto të hyrat nga instrumentet tjera financiare': '8',
            'Neto të hyrat (shpenzimet) tjera operative': '9', 'Gjithsej të hyrat': '10',
            'Provizionet për humbjet nga kreditë': '11', 'Provizione të tjera': '13',
            'Fitimi (humbja) para tatimit': '14', 'Fitimi/(humbja) para tatimit': '14', 'Fitimi para tatimit': '14',
            'Shpenzimet e tatimit në fitim': '15', 'Fitimi (humbja) neto': '16', 'Fitimi/(humbja) neto': '16',
            'Fitimi neto': '16', 'Të ardhurat tjera gjithëpërfshirëse': '17', 'Gjithsej të ardhurat gjithëpërfshirëse': '19',
            'Gjithsej te ardhurat gjithëpërfshirëse': '19', 'Gjithsejt te ardhurat gjitheperfshirese per vitin': '19',
            'Gjithsejt të ardhurat gjithëpërfshirëse për vitin': '19', 'Fitimi neto pas tatimit': '64',
            'Fitimi/(humbja) neto pas tatimit': '64'
        }
        full_is['CATEGORY_CODE'] = full_is['INCOME_CATEGORY'].map(is_map)
        full_is['STATEMENT_TYPE'] = 'INCOME_STATEMENT'
        full_is['DT_REPORT'] = pd.to_datetime(full_is['DT_REPORT'], format='%d.%m.%Y', errors='coerce')
    
    # Create unified DataFrame
    unified_dfs = []
    if not full_is.empty:
        is_unified = full_is.rename(columns={'INCOME_CATEGORY': 'ORIGINAL_CATEGORY', 'PREVIOUS_INCOME_VALUE': 'PREVIOUS_VALUE', 'CURRENT_INCOME_VALUE': 'CURRENT_VALUE'})
        is_unified['CATEGORY_TYPE'] = 'INCOME'
        unified_dfs.append(is_unified)
    
    if not full_bs.empty:
        bs_unified = full_bs.rename(columns={'BALANCE_CATEGORY': 'ORIGINAL_CATEGORY', 'PREVIOUS_BALANCE_VALUE': 'PREVIOUS_VALUE', 'CURRENT_BALANCE_VALUE': 'CURRENT_VALUE'})
        bs_unified['CATEGORY_TYPE'] = 'BALANCE'
        unified_dfs.append(bs_unified)
    
    if unified_dfs:
        final_df = pd.concat(unified_dfs, ignore_index=True)
        final_df['BANK_ID'] = 1
        final_df['CURR_ID'] = 1
        final_df['DATA_SOURCE'] = 'TEB Kosovo'
        final_df['EXTRACTION_DATE'] = datetime.now().strftime('%Y-%m-%d')
        final_df['REPORT_DATE'] = final_df['DT_REPORT'].dt.strftime('%Y-%m-%d')
        final_df = final_df.dropna(subset=['DT_REPORT']).sort_values(['DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_CODE'])
        
        # Reorder columns
        cols = ['BANK_ID', 'REPORT_DATE', 'DT_REPORT', 'STATEMENT_TYPE', 'CATEGORY_TYPE', 'CATEGORY_CODE', 
                'ORIGINAL_CATEGORY', 'PREVIOUS_VALUE', 'CURRENT_VALUE', 'CURR_ID', 'FILE_NAME', 'DATA_SOURCE', 'EXTRACTION_DATE']
        final_df = final_df[[c for c in cols if c in final_df.columns]]
        
        # Save outputs
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        final_df.to_csv(os.path.join(dirs['output'], f"teb_unified_{timestamp}.csv"), index=False)
        
        with pd.ExcelWriter(os.path.join(dirs['output'], f"teb_report_{timestamp}.xlsx"), engine='openpyxl') as writer:
            final_df.to_excel(writer, sheet_name='Unified', index=False)
            if not full_is.empty: full_is.to_excel(writer, sheet_name='Income_Statement', index=False)
            if not full_bs.empty: full_bs.to_excel(writer, sheet_name='Balance_Sheet', index=False)
        
        print(f"\n📊 UNIFIED: {len(final_df)} rows, {final_df['REPORT_DATE'].nunique()} dates")
        print(f"⏱️ Completed in {time.time()-start:.2f}s")
        
        return final_df, full_is, full_bs
    
    return pd.DataFrame(), full_is, full_bs

# Run
print("🚀 Starting extraction...")
final_df, income_df, balance_df = main(force=True, extract_only=False)

if not final_df.empty:
    print("\n" + "="*60)
    print("📊 FIRST 10 ROWS:")
    print(final_df.head(10))
    
    print("\n🔍 Check for Kërkesat ndaj bankave:")
    kerkesat = final_df[final_df['ORIGINAL_CATEGORY'].str.contains('Kërkesat ndaj bankave', na=False)]
    print(kerkesat[['REPORT_DATE', 'CURRENT_VALUE']] if not kerkesat.empty else "No data found")